# SWE015 · Real-Time Chat Moderation — training & evaluation on Colab

Runtime → *Change runtime type* → **T4 GPU**. Then *Runtime → Run all*.
Models are written to Google Drive so a disconnect does not lose them.

In [ ]:
!nvidia-smi -L || echo "No GPU - BERT fine-tuning will be slow"

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/MuhammetGoksenin/swe015-chat-moderation.git"
import os
if not os.path.exists("/content/swe015-chat-moderation"):
    !git clone -q {REPO_URL} /content/swe015-chat-moderation
%cd /content/swe015-chat-moderation
!pip install -q -r requirements.txt

## 2. Keep data + models on Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["DATA_DIR"] = "/content/drive/MyDrive/swe015/data"
os.environ["MODELS_DIR"] = "/content/drive/MyDrive/swe015/models"
os.environ["RESULTS_DIR"] = "/content/swe015-chat-moderation/results"

## 3. Data + classical baselines (Area B)

In [ ]:
!python train/00_prepare_data.py

In [ ]:
!python train/01_baselines.py

## 4. word2vec + feed-forward network (Area A)

In [ ]:
!python train/02_word2vec_ffnn.py

## 5. Fine-tune BERTurk (Area C) — ~10 min/epoch on a T4

In [ ]:
!python train/03_finetune_bert.py

## 6. Dynamic INT8 quantization (Area D)

In [ ]:
!python train/04_quantize.py

## 7. Evaluation on the held-out test set (CPU latency, ≥100 runs)
Latency is measured on the Colab **CPU** (all models are moved to CPU). The hardware line is written into the table.

In [ ]:
!python train/05_evaluate.py
from IPython.display import Markdown, display
display(Markdown(open("results/results.md", encoding="utf-8").read()))

In [ ]:
import pandas as pd
pd.read_csv("results/errors_bert_int8.csv")[["text", "label", "p_bert_int8", "error"]].head(10)  # failure examples for the slides

## 8. Upload weights to the Hugging Face Hub (R7: no weights in git)
Add a **write** token as a Colab secret named `HF_TOKEN` (key icon on the left) and set your HF username.

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_REPO_ID"] = "<your-hf-username>/swe015-chat-moderation-models"
!python train/06_push_to_hub.py

## 9. Download `results/` (commit it to GitHub) and try the app

In [ ]:
!zip -qr results.zip results && echo "results.zip ready (Files panel)"
os.environ["GRADIO_SHARE"] = "1"
!python app.py